# Evaluation and inference comparisons

greedy, sample10_frequency, sample5_union and direct_top5 share the unified evaluation CLI. model_ref and eval_input are lists. Use dev data for comparisons and official test batches only after selection. Multiple model refs compare SFT/DPO/base models under identical settings. The factoid workflow standardizes local generation, SFT selection, DPO selection and final official evaluation at max_seq_length=4096.

grounded_semantic adds a closed-snippet LLM judge for factoids. It reports semantic accuracy/MRR and the stricter grounded-semantic accuracy/MRR, where an answer must both match the accepted answer scope and be supported by the supplied snippets. Exact aliases are handled deterministically; only non-exact candidates use the judge API. semantic_judge_max_new_calls is one shared budget across all candidate models, and semantic_judge_cache_dir can reuse a prior cache. Grounded evaluation rejects locally truncated prompts because judging evidence that the candidate model could not see would be invalid; keep the 4096-token standard and use resource limits or sequential windows for a separate grounded evaluation.

gpt_candidate_pilot evaluates an OpenAI model as a candidate using the same prompt bundle and official scorer. model_ref is unified: use local paths/aliases normally and prefix API models with openai:, for example openai:gpt-4.1-mini-2025-04-14. The older openai_model option remains accepted. local_and_gpt_grounded_pilot adds API candidate generation and grounded judging. OpenAI candidate calls and judge calls have separate hard budgets. Increase limit and both budgets for a full run. Candidate and judge models are independently configurable; use a different judge family for a publication-quality comparison when possible.

compare_sampling creates one shared independent bank for first-five and frequency ranking, optionally adding greedy/direct-top-five runs. Its source is one prepared/raw file and model_ref one string. history_conditioned takes local base_model, adapter, prepared source and a frozen raw_generations.jsonl bank in the existing comparison schema.

openai_direct accepts raw/prepared sources and an explicit question_limit (default 10); it makes no retry calls. gpt_direct_vs_reasoning accepts a prepared source and paired prompt versions. Its question limit bounds questions; validation/provider retries may make additional calls. Both remain API-disabled until enabled. Official Java scoring is required.

Execution is **off by default**. Preview needs only Python and the repository. Select a kernel with the project's runtime dependencies before executing. Each method uses that kernel's Python in a separate worker process.


In [11]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/notebook_workflows").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_workflows import describe, plan, execute
from src.notebook_workflows.presets import PRESETS, configuration
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)


Project: C:\Users\Dustin\Projects\BioASQ-MRES
Python: c:\Users\Dustin\Projects\BioASQ-MRES\.venv\Scripts\python.exe


## Choose a preset

- **greedy**: local_evaluation
- **sample10_frequency**: local_evaluation
- **sample5_union**: local_evaluation
- **direct_top5**: local_evaluation
- **grounded_semantic**: local_evaluation
- **gpt_candidate_pilot**: local_evaluation
- **local_and_gpt_grounded_pilot**: local_evaluation
- **compare_sampling**: sampling_comparison
- **history_conditioned**: conditioned_sampling
- **gpt_direct_vs_reasoning**: gpt_reasoning
- **openai_direct**: openai_direct

Edit the configuration, then rerun the remaining cells. OVERRIDES uses the option names printed by describe(), with underscores. For CLI flags, True includes the named flag and False omits it. Missing inputs block execution.


In [ ]:
CATEGORY = 'evaluation'
PRESET = 'greedy'
OVERRIDES = {
    # Set input/model paths here using the fields printed below.
}
RUN = False
ALLOW_API = False

# Optional sequential comparisons: shared OVERRIDES plus per-variant changes.
VARIANTS = [{"label": "default", "overrides": {}}]

# Current experiment: compare the two saved SFT adapters under the repaired
# semantic rubric, using full GPT-4.1 as one independent judge.
DATA = (
    PROJECT_ROOT
    / "data/BioASQ_factoid_sft_prepared"
    / "split_first_train90_dev10_supported_train_seed3407"
)
GOLD_ONLY_SFT_05B = (
    PROJECT_ROOT
    / "Artifacts/notebook_runs/sft/answer_sft"
    / "20260927-143545-67f7a338/adapter"
)
GOLD_ONLY_SFT_3B = (
    PROJECT_ROOT
    / "Artifacts/notebook_runs/sft/answer_sft"
    / "20260927-143545-3fca1c3d/adapter"
)

OVERRIDES = {
    "eval_input": [str(DATA / "dev_prepared.json")],
    "max_seq_length": 4096,
    "model_ref": [str(GOLD_ONLY_SFT_05B), str(GOLD_ONLY_SFT_3B)],
    "score_backend": "bioasq_java",

    # The first 15 resources retain all 121 source-level extractable dev
    # questions. The longest rendered Qwen prompt is about 3,629 tokens, so the
    # candidate and judge see matching evidence without 4,096-token truncation.
    "max_resources": 15,
    "max_resource_chars": 0,

    # The v2 paired semantic_relation schema prevents contradictory combinations
    # such as semantic_label=equivalent with relation_type=narrower.
    "semantic_judge": True,
    "semantic_judge_model": "gpt-4.1-2025-04-14",
    "semantic_judge_api_key_file": str(PROJECT_ROOT / "open_ai_api.txt"),
    "semantic_judge_max_new_calls": 650,
    "semantic_judge_max_retries": 2,
    "semantic_judge_cache_dir": None,
    "limit": 160,
}

# Execution and API permission remain off until the plan has been reviewed.
RUN = True
ALLOW_API = True
VARIANTS = [{"label": "sft_05b_vs_3b_new_gpt41_judge", "overrides": {}}]

METHOD, PARAMETERS = configuration(CATEGORY, PRESET, OVERRIDES)
print(json.dumps({"method": METHOD, "parameters": PARAMETERS}, indent=2))


{
  "method": "local_evaluation",
  "parameters": {
    "chat_template": "qwen-2.5",
    "prompt_format": "chat",
    "prompt_file": "prompts/factoid_single_answer_aligned.json",
    "prompt": "factoid-single-answer-extractive-v1",
    "max_resources": 15,
    "max_resource_chars": 0,
    "max_seq_length": 4096,
    "eval_input": [
      "C:\\Users\\Dustin\\Projects\\BioASQ-MRES\\data\\BioASQ_factoid_sft_prepared\\split_first_train90_dev10_supported_train_seed3407\\dev_prepared.json"
    ],
    "model_ref": [
      "C:\\Users\\Dustin\\Projects\\BioASQ-MRES\\Artifacts\\notebook_runs\\sft\\answer_sft\\20260927-143545-67f7a338\\adapter",
      "C:\\Users\\Dustin\\Projects\\BioASQ-MRES\\Artifacts\\notebook_runs\\sft\\answer_sft\\20260927-143545-3fca1c3d\\adapter"
    ],
    "question_types": [
      "factoid"
    ],
    "num_generations": 1,
    "aggregation_strategy": "union",
    "temperature": 0.0,
    "top_p": 1.0,
    "max_new_tokens": 64,
    "seed": 3407,
    "score_backend": "bioas

## Inspect and preview

Options are read from source without importing the training stack. Unknown options and invalid choices are rejected. Outputs go under Artifacts/notebook_runs and cannot overwrite earlier runs. Rerun this cell to allocate fresh plans.


In [13]:
print(json.dumps(describe(METHOD), indent=2, default=str))
PLANS = []
for variant in VARIANTS:
    method, parameters = configuration(CATEGORY, PRESET, {**OVERRIDES, **variant["overrides"]})
    preview = plan(method, parameters, project_root=PROJECT_ROOT)
    PLANS.append(preview)
    print("\nVariant:", variant["label"])
    print("Method:", preview["method"])
    print("Output:", preview["run_dir"])
    print("Requires GPU:", preview["gpu"], "| Calls API:", preview["api"])
    print("Unresolved inputs:", preview["missing"] or "none")
    print(json.dumps(preview["parameters"], indent=2))


{
  "method": "local_evaluation",
  "description": "Evaluate local and/or OpenAI models with official and grounded-semantic metrics.",
  "defaults": {
    "chat_template": "qwen-2.5",
    "prompt_format": "chat",
    "prompt_file": "prompts/factoid_single_answer_aligned.json",
    "prompt": "factoid-single-answer-extractive-v1",
    "max_resources": 0,
    "max_resource_chars": 0,
    "max_seq_length": 4096,
    "eval_input": null,
    "model_ref": null,
    "question_types": [
      "factoid"
    ],
    "num_generations": 1,
    "aggregation_strategy": "union",
    "temperature": 0.0,
    "top_p": 1.0,
    "max_new_tokens": 64,
    "seed": 3407,
    "score_backend": "bioasq_java",
    "max_factoid_answers": 10000,
    "openai_model": null,
    "api_key_file": "open_ai_api.txt",
    "max_new_api_calls": 0,
    "openai_cache_dir": null,
    "semantic_judge": false,
    "semantic_judge_model": "gpt-4.1-mini-2025-04-14",
    "semantic_judge_api_key_file": "open_ai_api.txt",
    "semantic_

## Execute reviewed plans

Set RUN=True, then rerun configuration and preview. API methods also need ALLOW_API=True. Plans cannot be changed after preview. Each run retains its configuration, input hashes, status and log. A repeated execution of the same plan refuses to overwrite its directory; create a fresh plan instead.


In [14]:
RESULTS = []
for preview in PLANS:
    if RUN:
        result = execute(preview, run=True, allow_api=ALLOW_API)
        RESULTS.append(str(result))
    else:
        print("Preview only:", preview["method"], "- set RUN=True to execute.")


Preview only: local_evaluation - set RUN=True to execute.


## Inspect outputs

Use these explicit paths in the next notebook. Historical commands, plots and outputs are preserved in reproducibility/notebook_archive.zip; notebooks/MIGRATION.md maps old notebooks to these workflows.


In [15]:
for run in RESULTS:
    path = Path(run)
    print("\nRun:", path)
    print((path / "status.json").read_text(encoding="utf-8"))
    print("Log:", path / "run.log")
    print("Outputs:", [p.name for p in path.iterdir()
                      if p.name not in {"run.log", "plan.json", "input_hashes.json"}])
